# Exercise: Is your house worth it's weight in Dubloons?

You'd need pirate treasure to but a house in London in 2025, so let's find out if you'd be better of burying your money as Dubloons or buying a house with it. (Data restricted to post January 2023 to keep Trino from running out of memory on my ailing M1)

![](pirate.jpg)



In [17]:
import polars as pl
from utils import catalog, engine
from pyiceberg.schema import Schema, NestedField
from pyiceberg.types import DecimalType, DateType, StringType
from pyiceberg.partitioning import PartitionSpec, YearTransform, PartitionField
from IPython.display import display
pl.Config.set_thousands_separator(",")

polars.config.Config

In [18]:
sql = """
-- Convert USD gold prices into GBP denominated prices
with gold_prices as (
    select commodities.gold.date as gold_date, 
           commodities.gold.price * fx.rates.exchange_rate as gold_price
    from commodities.gold
    join fx.rates on fx.rates.date = commodities.gold.date
    where commodities.gold.date >= DATE '2023-01-01'  -- Add date filter if appropriate
), 
filtered_profits as (
    select * from housing.profits 
    where first_day >= DATE '2023-01-01'  -- Match the date filter
),
gold_purchase as (
    select address_id, 
           cast(first_price as DOUBLE) / cast(gold_price as DOUBLE) as purchased_gold,
           first_day,
           last_day,
           cast(gold_price as DOUBLE) as gold_price_at_purchase  -- Cast to DOUBLE
    from filtered_profits
    join gold_prices on gold_date = filtered_profits.first_day
), 
gold_sell as (
    select gp.address_id,
           cast((gp.purchased_gold * cast(gold_prices.gold_price as DOUBLE)) - cast(fp.first_price as DOUBLE) as DOUBLE) as gold_profit,
           cast(fp.profit as DOUBLE) as house_profit,  -- Cast to DOUBLE
           cast(fp.first_price as DOUBLE) as first_price,  -- Cast to DOUBLE
           cast(fp.last_price as DOUBLE) as last_price,   -- Cast to DOUBLE
           gp.gold_price_at_purchase,
           cast(gold_prices.gold_price as DOUBLE) as gold_price_at_sale  -- Cast to DOUBLE
    from gold_purchase gp
    join filtered_profits fp on fp.address_id = gp.address_id
    join gold_prices on gold_prices.gold_date = gp.last_day
)
select * from gold_sell
"""

In [19]:
gold_vs_house_profits = pl.read_database(sql, engine)
gold_vs_house_profits

address_id,gold_profit,house_profit,first_price,last_price,gold_price_at_purchase,gold_price_at_sale
str,f64,f64,f64,f64,f64,f64
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.389114",0.0,"420,000.0","420,000.0","1,575.53076","1,592.307871"
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.389114",0.0,"420,000.0","420,000.0","1,575.53076","1,592.307871"
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.389114",0.0,"420,000.0","420,000.0","1,575.53076","1,592.307871"
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.389114",0.0,"420,000.0","420,000.0","1,575.53076","1,592.307871"
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.389114",0.0,"420,000.0","420,000.0","1,575.53076","1,592.307871"
…,…,…,…,…,…,…
"""523EA4049BE7C657617E05D55E84B6…","10,636.742109","747,275.0","212,725.0","960,000.0","1,536.985928","1,613.83878"
"""523EA4049BE7C657617E05D55E84B6…","10,636.742109","747,275.0","212,725.0","960,000.0","1,536.985928","1,613.83878"
"""523EA4049BE7C657617E05D55E84B6…","10,636.742109","747,275.0","212,725.0","960,000.0","1,536.985928","1,613.83878"


In [21]:
with pl.Config(set_tbl_rows=100):
    # Constants for doubloon calculations
    DOUBLOON_FINE_GOLD_GRAMS = 6.2  # grams of fine gold per doubloon
    TROY_OUNCE_TO_GRAMS = 31.1035   # conversion factor
    
    summary_df = (
        gold_vs_house_profits
        # .filter(pl.col("county") == "KENT")  # Add this line to filter by county
        .with_columns([
            # Calculate house prices in grams of gold
            (pl.col("first_price") / pl.col("gold_price_at_purchase") * TROY_OUNCE_TO_GRAMS).alias("first_price_in_gold_grams"),
            (pl.col("last_price") / pl.col("gold_price_at_sale") * TROY_OUNCE_TO_GRAMS).alias("last_price_in_gold_grams"),
            
            # Calculate number of doubloons
            (pl.col("first_price") / pl.col("gold_price_at_purchase") * TROY_OUNCE_TO_GRAMS / DOUBLOON_FINE_GOLD_GRAMS).alias("first_price_doubloons"),
            (pl.col("last_price") / pl.col("gold_price_at_sale") * TROY_OUNCE_TO_GRAMS / DOUBLOON_FINE_GOLD_GRAMS).alias("last_price_doubloons")
        ])
        .select([
            # Beginning of sample averages
            pl.col("first_price").mean().alias("avg_first_price_gbp"),
            pl.col("first_price_in_gold_grams").mean().alias("avg_first_price_gold_grams"),
            pl.col("first_price_doubloons").mean().alias("avg_first_price_doubloons"),
            
            # End of sample averages
            pl.col("last_price").mean().alias("avg_last_price_gbp"),
            pl.col("last_price_in_gold_grams").mean().alias("avg_last_price_gold_grams"),
            pl.col("last_price_doubloons").mean().alias("avg_last_price_doubloons")
        ])
        .with_columns([
            # Calculate deltas
            (pl.col("avg_last_price_gbp") - pl.col("avg_first_price_gbp")).alias("delta_price_gbp"),
            (pl.col("avg_last_price_gold_grams") - pl.col("avg_first_price_gold_grams")).alias("delta_price_gold_grams"),
            (pl.col("avg_last_price_doubloons") - pl.col("avg_first_price_doubloons")).alias("delta_price_doubloons"),
            
            # Calculate percentage changes
            ((pl.col("avg_last_price_gbp") - pl.col("avg_first_price_gbp")) / pl.col("avg_first_price_gbp") * 100).alias("pct_change_gbp"),
            ((pl.col("avg_last_price_doubloons") - pl.col("avg_first_price_doubloons")) / pl.col("avg_first_price_doubloons") * 100).alias("pct_change_doubloons")
        ])
    )
    display(summary_df)

avg_first_price_gbp,avg_first_price_gold_grams,avg_first_price_doubloons,avg_last_price_gbp,avg_last_price_gold_grams,avg_last_price_doubloons,delta_price_gbp,delta_price_gold_grams,delta_price_doubloons,pct_change_gbp,pct_change_doubloons
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"238,095.659404","4,744.141182",765.184062,"306,124.663991","6,068.714527",978.824924,"68,029.004587","1,324.573345",213.640862,28.572131,27.920192
